In [1]:
import numpy as np
import pandas as pd

from statsmodels.formula.api import logit, probit, ols

# Не показывать FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

from IPython.display import Markdown, display

# загружен датасеты в виде словаря
path_to_datasets = '../datasets/'
df_dict = {'loanapp': pd.read_csv(path_to_datasets+'loanapp.csv'), 
		'TableF5-1':pd.read_csv(path_to_datasets+'TableF5-1.csv'), 
		'SwissLabour': pd.read_csv(path_to_datasets+'SwissLabour.csv')}

## Предельные значения/эффекты

### Формулы для предельных эффектов

__для probit модели__ $P(y=1)=\Phi(x^\top\beta)$ формула для предельных значений

$$
\begin{aligned}
	\frac{\partial P(y=1)}{\partial x_j}&=\phi(x^\top\beta)\beta_j & \phi(z)=\frac{1}{\sqrt{2\pi}}exp\left(-z^2/2\right)
\end{aligned}
$$

__для logit модели__ $P(y=1)=\Lambda(x^\top\beta)$ формула для предельных значений

$$
\begin{aligned}
	\frac{\partial P(y=1)}{\partial x_j}&=\lambda(x^\top\beta)\beta_j & \lambda(z)=\frac{exp(z)}{(1+exp(z))^2}
\end{aligned}
$$

### Средние предельные эффекты

Обычно рассчитываются

- предельные значения для каждого регрессора в средней точке $x_{mean}=\sum_i x_i/n$: 

$$
\begin{aligned}
	probit:&\ \phi(x_{mean}^\top\beta)\beta_j & logit:&\ \lambda(x_{mean}^\top\beta)\beta_j
\end{aligned}
$$

- среднее по всей выборке предельное значения для каждого регрессора:

$$
\begin{aligned}
	probit:&\ \frac 1n \sum_{i}\phi(x_i^\top\beta)\beta_j & logit:&\ \frac 1n \sum_{i}\lambda(x_i^\top\beta)\beta_j
\end{aligned}
$$

In [2]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =['I(appinc/100)+mortno+unem+dep+male',
			'WA+np.log(FAMINC)+WE+KL6+K618+CIT+UN',
			'income+age+I(age**2)+youngkids+oldkids']

mod_name = ['Approve equation', 'labour force equation', 'swiss labour force equation']

sign_level = [0.05, 0.01, 0.10]

digits = 3

for i, df_name in enumerate(df_names):
	mod_logit  = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_probit = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_ols = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res_logit = mod_logit.fit(disp=False)
	res_probit = mod_probit.fit(disp=False)
	res_ols = mod_ols.fit(cov_type='HC3')

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'Для датасет `{df_name}` оцените __probit__  и __logit__ регрессии `{endog_var[i]}` на `{exog_vars[i]}`. Для каждой регрессии вычислите предельные значения и дайте их интерпретацию:\n'
	f'- в средней точке для каждого регрессора \n'
	f'- среднее по выборке предельное значение для каждого регрессора.\n\n '
	f'Тестируйте значимость предельных эффектов при уровне значимости {sign_level[i]*100}%. Сравните с предельными эффектами в LPM-модели. __Ответ округлите до {digits} десятичных знаков__'))

	df_margeff = pd.DataFrame({'Регрессор': res_logit.params.index[1:], 
								'LPM-эффект': res_ols.params[1:].round(digits),
								'В ср.точке (logit)': res_logit.get_margeff(at='mean', dummy=True).margeff.round(digits),
								'В ср.точке (probit)': res_probit.get_margeff(at='mean', dummy=True).margeff.round(digits),
								'Ср.по выборке (logit)': res_logit.get_margeff(at='overall', dummy=True).margeff.round(digits),
								'Ср.по выборке (probit)': res_probit.get_margeff(at='overall', dummy=True).margeff.round(digits)
								})

	df_margeff_stats = pd.DataFrame({'Регрессор': res_logit.params.index[1:], 
									'LPM-эффект': res_ols.tvalues[1:].round(digits),
									'В ср.точке (logit)': res_logit.get_margeff(at='mean', dummy=True).tvalues.round(digits),
									'В ср.точке (probit)': res_probit.get_margeff(at='mean', dummy=True).tvalues.round(digits),
									'Ср.по выборке (logit)': res_logit.get_margeff(at='overall', dummy=True).tvalues.round(digits),
									'Ср.по выборке (probit)': res_probit.get_margeff(at='overall', dummy=True).tvalues.round(digits)
									})

	df_margeff_sign = pd.DataFrame({'Регрессор':res_logit.params.index[1:], 
								'LPM-эффект': res_ols.pvalues[1:].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'В ср.точке (logit)': res_logit.get_margeff(at='mean', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'В ср.точке (probit)': res_probit.get_margeff(at='mean', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'Ср.по выборке (logit)': res_logit.get_margeff(at='overall', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'Ср.по выборке (probit)': res_probit.get_margeff(at='overall', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим')
								})

	display(Markdown(f'''__Ответ__: предельные эффекты\n'''))
	display(Markdown(df_margeff.to_markdown(index=False)))
	display(Markdown(f'''их тестовые статистики\n'''))
	display(Markdown(df_margeff_stats.to_markdown(index=False)))
	display(Markdown(f'''значимость предельных эффектов\n'''))
	display(Markdown(df_margeff_sign.to_markdown(index=False)))



### Approve equation

Для датасет `loanapp` оцените __probit__  и __logit__ регрессии `approve` на `I(appinc/100)+mortno+unem+dep+male`. Для каждой регрессии вычислите предельные значения и дайте их интерпретацию:
- в средней точке для каждого регрессора 
- среднее по выборке предельное значение для каждого регрессора.

 Тестируйте значимость предельных эффектов при уровне значимости 5.0%. Сравните с предельными эффектами в LPM-модели. __Ответ округлите до 3 десятичных знаков__

__Ответ__: предельные эффекты


| Регрессор       |   LPM-эффект |   В ср.точке (logit) |   В ср.точке (probit) |   Ср.по выборке (logit) |   Ср.по выборке (probit) |
|:----------------|-------------:|---------------------:|----------------------:|------------------------:|-------------------------:|
| I(appinc / 100) |       -0.014 |               -0.011 |                -0.011 |                  -0.011 |                   -0.011 |
| mortno          |        0.079 |                0.077 |                 0.077 |                   0.078 |                    0.077 |
| unem            |       -0.008 |               -0.007 |                -0.007 |                  -0.007 |                   -0.007 |
| dep             |       -0.012 |               -0.011 |                -0.011 |                  -0.011 |                   -0.011 |
| male            |        0.019 |                0.019 |                 0.019 |                   0.019 |                    0.019 |

их тестовые статистики


...

| Регрессор   |   LPM-эффект |   В ср.точке (logit) |   В ср.точке (probit) |   Ср.по выборке (logit) |   Ср.по выборке (probit) |
|:------------|-------------:|---------------------:|----------------------:|------------------------:|-------------------------:|
| income      |       -0.258 |               -0.33  |                -0.318 |                  -0.279 |                   -0.278 |
| age         |        0.795 |                0.965 |                 0.934 |                   0.817 |                    0.815 |
| I(age ** 2) |       -0.112 |               -0.136 |                -0.131 |                  -0.115 |                   -0.115 |
| youngkids   |       -0.228 |               -0.264 |                -0.253 |                  -0.224 |                   -0.221 |
| oldkids     |       -0.053 |               -0.062 |                -0.06  |                  -0.053 |                   -0.053 |

их тестовые статистики


| Регрессор   |   LPM-эффект |   В ср.точке (logit) |   В ср.точке (probit) |   Ср.по выборке (logit) |   Ср.по выборке (probit) |
|:------------|-------------:|---------------------:|----------------------:|------------------------:|-------------------------:|
| income      |       -7.533 |               -6.249 |                -6.428 |                  -6.788 |                   -6.898 |
| age         |        6.405 |                5.849 |                 5.95  |                   6.288 |                    6.323 |
| I(age ** 2) |       -7.527 |               -6.633 |                -6.79  |                  -7.295 |                   -7.359 |
| youngkids   |       -7.029 |               -6.458 |                -6.69  |                  -7.082 |                   -7.232 |
| oldkids     |       -2.965 |               -3.052 |                -3.055 |                  -3.112 |                   -3.104 |

значимость предельных эффектов


| Регрессор   | LPM-эффект   | В ср.точке (logit)   | В ср.точке (probit)   | Ср.по выборке (logit)   | Ср.по выборке (probit)   |
|:------------|:-------------|:---------------------|:----------------------|:------------------------|:-------------------------|
| income      | Значим       | Значим               | Значим                | Значим                  | Значим                   |
| age         | Значим       | Значим               | Значим                | Значим                  | Значим                   |
| I(age ** 2) | Значим       | Значим               | Значим                | Значим                  | Значим                   |
| youngkids   | Значим       | Значим               | Значим                | Значим                  | Значим                   |
| oldkids     | Значим       | Значим               | Значим                | Значим                  | Значим                   |